# 01 — Chargement et qualité des données

**Objectif** : charger le dataset de manière fiable et reproductible, puis évaluer sa qualité **avant** toute analyse.

**Questions traitées**
1. Les données respectent-elles le schéma attendu (colonnes, types, cible binaire) ?
2. Y a-t-il des valeurs manquantes, des doublons ou des valeurs aberrantes évidentes ?
3. Les soldes sont-ils comptablement cohérents avec les montants ?
4. Quelles particularités des données devront être prises en compte dans la suite ?

> Toute la logique est implémentée et testée dans `src/data.py` ; ce notebook se contente de l'appeler et d'interpréter les résultats.

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import pandas as pd

from src.config import RAW_CSV_PATH, CLEAN_PARQUET_PATH
from src.data import load_raw, save_parquet, quality_report, missing_values_report, balance_consistency

pd.set_option("display.float_format", lambda x: f"{x:,.2f}")
pd.set_option("display.max_columns", 20)

## 1. Chargement

Le CSV brut est chargé avec des **types optimisés** (`int16` pour `step`, `category` pour `type`, `int8` pour les indicateurs binaires).
Les montants et soldes restent en `float64` pour éviter des erreurs d'arrondi sur les futurs calculs d'écarts de solde.

La colonne `oldbalanceOrg` est renommée en `oldbalanceOrig` pour harmoniser le nommage avec `newbalanceOrig`.

In [2]:
%%time
df = load_raw(RAW_CSV_PATH)
print(f"{df.shape[0]:,} lignes x {df.shape[1]} colonnes")

6,362,620 lignes x 11 colonnes
CPU times: total: 2 s
Wall time: 4.31 s


In [3]:
df.head()

,step,type,amount,nameOrig,oldbalanceOrig,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud
0,1,PAYMENT,"9,839.64",C1231006815,"170,136.00","160,296.36",M1979787155,0.00,0.00,0,0
1,1,PAYMENT,"1,864.28",C1666544295,"21,249.00","19,384.72",M2044282225,0.00,0.00,0,0
2,1,TRANSFER,181.00,C1305486145,181.00,0.00,C553264065,0.00,0.00,1,0
3,1,CASH_OUT,181.00,C840083671,181.00,0.00,C38997010,"21,182.00",0.00,1,0
4,1,PAYMENT,"11,668.14",C2048537720,"41,554.00","29,885.86",M1230701703,0.00,0.00,0,0


In [4]:
df.dtypes.to_frame('dtype')

,dtype
step,int16
type,category
amount,float64
nameOrig,string
oldbalanceOrig,float64
newbalanceOrig,float64
nameDest,string
oldbalanceDest,float64
newbalanceDest,float64
isFraud,int8


## 2. Synthèse de qualité

In [5]:
report = quality_report(df)
pd.Series(report, name="valeur").to_frame()

,valeur
n_rows,6362620
n_columns,11
memory_mb,497.30
n_missing_total,0
n_duplicated_rows,0
n_negative_amount,0
n_zero_amount,16
n_negative_balance,0
step_min,1
step_max,743


## 3. Valeurs manquantes

In [6]:
missing_values_report(df)

,n_missing,pct_missing
step,0,0.00
type,0,0.00
amount,0,0.00
nameOrig,0,0.00
oldbalanceOrig,0,0.00
newbalanceOrig,0,0.00
nameDest,0,0.00
oldbalanceDest,0,0.00
newbalanceDest,0,0.00
isFraud,0,0.00


## 4. Statistiques descriptives des variables numériques

In [7]:
num_cols = ["amount", "oldbalanceOrig", "newbalanceOrig", "oldbalanceDest", "newbalanceDest"]
df[num_cols].describe(percentiles=[0.25, 0.5, 0.75, 0.99]).T

,count,mean,std,min,25%,50%,75%,99%,max
amount,"6,362,620.00","179,861.90","603,858.23",0.00,"13,389.57","74,871.94","208,721.48","1,615,979.47","92,445,516.64"
oldbalanceOrig,"6,362,620.00","833,883.10","2,888,242.67",0.00,0.00,"14,208.00","107,315.18","16,027,256.13","59,585,040.37"
newbalanceOrig,"6,362,620.00","855,113.67","2,924,048.50",0.00,0.00,0.00,"144,258.41","16,176,160.56","49,585,040.37"
oldbalanceDest,"6,362,620.00","1,100,701.67","3,399,180.11",0.00,0.00,"132,705.66","943,036.71","12,371,819.15","356,015,889.35"
newbalanceDest,"6,362,620.00","1,224,996.40","3,674,128.94",0.00,0.00,"214,661.44","1,111,909.25","13,137,866.94","356,179,278.92"


## 5. Montants nuls et cas particuliers

In [8]:
zero_amount = df[df["amount"] == 0]
print(f"Transactions à montant nul : {len(zero_amount)}")
zero_amount[["step", "type", "amount", "oldbalanceOrig", "newbalanceOrig", "isFraud"]]

Transactions à montant nul : 16


,step,type,amount,oldbalanceOrig,newbalanceOrig,isFraud
2736447,212,CASH_OUT,0.00,0.00,0.00,1
3247298,250,CASH_OUT,0.00,0.00,0.00,1
3760289,279,CASH_OUT,0.00,0.00,0.00,1
5563714,387,CASH_OUT,0.00,0.00,0.00,1
5996408,425,CASH_OUT,0.00,0.00,0.00,1
5996410,425,CASH_OUT,0.00,0.00,0.00,1
6168500,554,CASH_OUT,0.00,0.00,0.00,1
6205440,586,CASH_OUT,0.00,0.00,0.00,1
6266414,617,CASH_OUT,0.00,0.00,0.00,1
6281483,646,CASH_OUT,0.00,0.00,0.00,1


## 6. Cohérence comptable des soldes

Pour une transaction cohérente on attend :
- **émetteur** : `oldbalanceOrig − amount = newbalanceOrig` (sauf `CASH_IN` où l'émetteur est crédité) ;
- **destinataire** : `oldbalanceDest + amount = newbalanceDest`.

Le tableau donne le **pourcentage de transactions cohérentes** (tolérance de 0,01) par type.

In [9]:
balance_consistency(df)

,pct_orig_coherent,pct_dest_coherent
type,,
CASH_IN,92.78,62.43
CASH_OUT,11.00,77.32
DEBIT,69.92,76.75
PAYMENT,45.81,0.00
TRANSFER,4.53,77.33


Les soldes des **marchands** (destinataires `M…`) sont-ils renseignés ?

In [10]:
is_merchant = df["nameDest"].str.startswith("M")
merchants = df.loc[is_merchant, ["oldbalanceDest", "newbalanceDest"]]
print(f"Transactions vers un marchand : {is_merchant.sum():,}")
print(f"Dont soldes destinataire tous deux nuls : {(merchants == 0).all(axis=1).mean():.2%}")
print("Types concernés :", df.loc[is_merchant, "type"].astype(str).value_counts().to_dict())

Transactions vers un marchand : 2,151,495
Dont soldes destinataire tous deux nuls : 100.00%


Types concernés : {'PAYMENT': 2151495}


### 6.1 Pourquoi l'émetteur est-il si souvent incohérent ?

Pour `TRANSFER` et `CASH_OUT`, moins de 12 % des transactions sont cohérentes côté émetteur. Hypothèse : la transaction est exécutée **même quand le solde est insuffisant** (`oldbalanceOrig < amount`), et le solde final est alors ramené à 0.

In [11]:
rows = []
for t in ["TRANSFER", "CASH_OUT", "PAYMENT", "DEBIT"]:
    s = df[df["type"] == t]
    enough = s["oldbalanceOrig"] >= s["amount"]
    coherent = (s["oldbalanceOrig"] - s["amount"] - s["newbalanceOrig"]).abs() <= 0.01
    rows.append({
        "type": t,
        "% solde insuffisant": enough.eq(False).mean() * 100,
        "% solde initial nul": s["oldbalanceOrig"].eq(0).mean() * 100,
        "% cohérent si solde suffisant": coherent[enough].mean() * 100,
    })
pd.DataFrame(rows).set_index("type")

,% solde insuffisant,% solde initial nul,% cohérent si solde suffisant
type,,,
TRANSFER,95.23,53.06,95.00
CASH_OUT,88.54,45.85,96.04
PAYMENT,51.18,35.99,93.83
DEBIT,28.45,14.86,97.72


### 6.2 Destinataires clients sans solde renseigné

In [12]:
dest_zero = (df["oldbalanceDest"] == 0) & (df["newbalanceDest"] == 0) & ~is_merchant
print(f"Destinataires clients avec soldes avant/après nuls : {dest_zero.sum():,}")
pd.crosstab(df.loc[dest_zero, "type"].astype(str), df.loc[dest_zero, "isFraud"], margins=True)

Destinataires clients avec soldes avant/après nuls : 165,787


isFraud,0,1,All
type,,,
CASH_IN,160005,0,160005
CASH_OUT,1600,8,1608
TRANSFER,106,4068,4174
All,161711,4076,165787


### 6.3 Plafonnement des montants frauduleux

In [13]:
fraud = df[df["isFraud"] == 1]
print(f"Montant frauduleux maximal : {fraud['amount'].max():,.0f}")
print(f"Fraudes à exactement 10 000 000 : {(fraud['amount'] == 10_000_000).sum()}")

Montant frauduleux maximal : 10,000,000
Fraudes à exactement 10 000 000 : 287


## 7. Cible et règle métier

In [14]:
pd.crosstab(df["type"], df["isFraud"], margins=True, margins_name="Total")

isFraud,0,1,Total
type,,,
CASH_IN,1399284,0,1399284
CASH_OUT,2233384,4116,2237500
DEBIT,41432,0,41432
PAYMENT,2151495,0,2151495
TRANSFER,528812,4097,532909
Total,6354407,8213,6362620


In [15]:
pd.crosstab(df["isFraud"], df["isFlaggedFraud"], margins=True, margins_name="Total")

isFlaggedFraud,0,1,Total
isFraud,,,
0,6354407,0,6354407
1,8197,16,8213
Total,6362604,16,6362620


## 8. Sauvegarde au format Parquet

Le Parquet est un format **colonnaire compressé** qui conserve les types : le relire est bien plus rapide que le CSV.
Les notebooks suivants utiliseront `load_data()`, qui lit directement ce cache.

In [16]:
%%time
path = save_parquet(df, CLEAN_PARQUET_PATH)
csv_mb = RAW_CSV_PATH.stat().st_size / 1024**2
pq_mb = path.stat().st_size / 1024**2
print(f"CSV : {csv_mb:,.0f} Mo  ->  Parquet : {pq_mb:,.0f} Mo  (÷{csv_mb / pq_mb:.1f})")

CSV : 471 Mo  ->  Parquet : 257 Mo  (÷1.8)
CPU times: total: 2.88 s
Wall time: 3.1 s


## 9. Conclusions

### Qualité générale : bonne
- **6 362 620 transactions**, schéma conforme, **aucune valeur manquante**, **aucun doublon**, aucun montant ni solde négatif.
- La période couvre les steps 1 à 743 (~31 jours).

### Particularités à prendre en compte

| # | Constat | Conséquence pour la suite |
|---|---|---|
| 1 | **Déséquilibre extrême** : 8 213 fraudes (0,13 %), uniquement en `TRANSFER` et `CASH_OUT`. | Métriques adaptées (PR-AUC), gestion du déséquilibre, filtrage possible sur ces 2 types (à valider en EDA). |
| 2 | **Règle métier quasi inutile** : 16 alertes, toutes justes, mais 99,8 % des fraudes ratées. | Baseline métier facile à battre ; `isFlaggedFraud` ne sera **pas** utilisée comme variable explicative. |
| 3 | **Soldes émetteurs souvent incohérents** : la transaction passe même avec un solde insuffisant, le solde final étant ramené à 0. Quand le solde est suffisant, ~95 % des transactions sont cohérentes. | Ces écarts sont un artefact du simulateur ; ils seront transformés en variables d'« erreur de solde » et leur pouvoir discriminant testé (H4). |
| 4 | **Soldes marchands non renseignés** : 100 % des destinataires `M…` (tous des `PAYMENT`) ont des soldes à 0. | Les soldes destinataires ne sont pas informatifs pour les paiements marchands. |
| 5 | **Destinataires clients à soldes nuls** : 165 787 transactions vers des clients ont des soldes avant/après nuls. Parmi les `TRANSFER` concernés, **4 068 sur 4 174 sont des fraudes** (97 %). | Signal très fort, à analyser en EDA ; il faudra vérifier qu'il ne s'agit pas d'une **fuite de données** propre au simulateur. |
| 6 | **16 transactions à montant nul**, toutes frauduleuses (`CASH_OUT`). | Cas marginaux conservés ; signalés comme artefacts de simulation. |
| 7 | **Montants frauduleux plafonnés à 10 000 000** (287 fraudes à ce montant exact). | Artefact du simulateur : une fraude plus importante est découpée en plusieurs transactions. |
| 8 | **Émetteurs quasi uniques** : 6,35 M d'émetteurs distincts pour 6,36 M de transactions. | Impossible de construire un historique par émetteur ; les variables comportementales porteront plutôt sur les destinataires (2,7 M distincts). |

### Données prêtes
Le jeu de données typé est sauvegardé dans `data/processed/transactions.parquet` (lecture rapide via `load_data()`).

**Étape suivante** : analyse exploratoire approfondie et tests statistiques des hypothèses H1 à H5.